# Implementation of the LEEP score

LEEP: A New Measure to Evaluate Transferability of Learned Representations, Cuong V. Nguyen and Tal Hassner and Matthias Seeger and Cedric Archambeau, 2020, https://arxiv.org/abs/2002.12462

In [2]:
%xmode minimal

import os
import json

os.environ["CUDA_VISIBLE_DEVICES"] = "-1"  # disable GPU devices
os.environ["TFDS_DATA_DIR"] = os.path.expanduser("~/tensorflow_datasets")  # default location of tfds database

import os
os.environ["KERAS_BACKEND"] = "tensorflow"

import keras
from keras import layers, models
from keras.applications import VGG16

import tensorflow as tf
import tensorflow_datasets as tfds

import librosa
import librosa.display

import numpy as np
from matplotlib import pyplot as plt

from pathlib import Path

from IPython.display import Audio

# Turn off logging for TF
import logging
tf.get_logger().setLevel(logging.ERROR)

# from tensorflow.python.client import device_lib
# print(device_lib.list_local_devices())

import dpmhm
# dpmhm.datasets.get_dataset_list()

from dpmhm.datasets import preprocessing, feature, utils

Exception reporting mode: Minimal


2024-07-04 13:48:51.844441: I external/local_tsl/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2024-07-04 13:48:51.849501: I external/local_tsl/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2024-07-04 13:48:51.921429: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2024-07-04 13:48:53.426786: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


In [3]:
dataset_name = 'CWRU'

ds_all, ds_info = tfds.load(
    dataset_name,
    with_info=True,
)

ds0 = ds_all['train']

2024-07-04 13:48:56.873637: E external/local_xla/xla/stream_executor/cuda/cuda_driver.cc:282] failed call to cuInit: CUDA_ERROR_NO_DEVICE: no CUDA-capable device is detected
2024-07-04 13:48:56.873729: I external/local_xla/xla/stream_executor/cuda/cuda_diagnostics.cc:134] retrieving CUDA diagnostic information for host: is230816
2024-07-04 13:48:56.873758: I external/local_xla/xla/stream_executor/cuda/cuda_diagnostics.cc:141] hostname: is230816
2024-07-04 13:48:56.874015: I external/local_xla/xla/stream_executor/cuda/cuda_diagnostics.cc:165] libcuda reported version is: 535.183.1
2024-07-04 13:48:56.874057: I external/local_xla/xla/stream_executor/cuda/cuda_diagnostics.cc:169] kernel reported version is: 535.183.1
2024-07-04 13:48:56.874075: I external/local_xla/xla/stream_executor/cuda/cuda_diagnostics.cc:248] kernel version seems to match DSO: 535.183.1


In [4]:
from dpmhm.datasets import transformer, feature

compactor = transformer.DatasetCompactor(ds0,
                                         channels=['DE', 'FE', 'BA'],
                                         keys=['FaultLocation', 'FaultComponent', 'FaultSize'],
                                         resampling_rate=12000)


_func = lambda x, sr: feature.spectral_features(x, sr, 'spectrogram',
                                                time_window=0.025, hop_step=0.0125, n_fft=512,
                                                normalize=False, to_db=True)[0]


extractor = transformer.FeatureExtractor(compactor.dataset, _func)

window = transformer.WindowSlider(extractor.dataset, window_size=(64,64), hop_size=(32,32))

compactor.dataset.element_spec

labels = list(compactor.full_label_dict.keys())

preproc = preprocessing.get_mapping_supervised(labels)

ds_window = window.dataset.map(preproc, num_parallel_calls=tf.data.AUTOTUNE)

eles = list(ds_window.take(10).as_numpy_iterator())
input_shape = eles[0][0].shape

ds_window = ds_window.map(lambda x,y: (tf.ensure_shape(x, input_shape), y), num_parallel_calls=tf.data.AUTOTUNE)
splits = {'train':0.99, 'test':0.01}
ds_split = utils.split_dataset(ds_window, splits, labels=[i+1 for i in range(30)])

batch_size = 32

ds_size = 20000  
ds_train = ds_split['test'].shuffle(ds_size, reshuffle_each_iteration=True).cache().batch(1).prefetch(tf.data.AUTOTUNE)
images_target=ds_train.map(lambda x,_: x)
labels_target=ds_train.map(lambda _,x : x)

2024-07-04 13:49:00.849711: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
2024-07-04 13:49:01.586626: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
2024-07-04 13:49:02.305890: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
2024-07-04 13:49:03.435097: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
2024-07-04 13:49:04.582691: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
2024-07-04 13:49:23.408204: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
2024-07-04 13:49:23.439431: W tensorflow/core/framework/local_rendezvous.cc:404] L

In [5]:
def LEEP(model, images_targets, labels_target, num_classes_source, num_classes_target):
    logits = model.predict(images_targets)

    probabilities = tf.nn.softmax(logits, axis=1).numpy()

    # p(y,z)
    prob_y_z = np.zeros((num_classes_target, num_classes_source))
    for y in range(num_classes_target):
        for z in range(num_classes_source):
            for i in range(len(probabilities)):
                if y+1 == labels_target[i]:
                    prob_y_z[y,z] += probabilities[i,z]
            prob_y_z[y,z] /= len(probabilities)

    # p(z)
    prob_z=np.zeros(num_classes_source)
    for z in range(num_classes_source):
        for i in range(len(probabilities)):
            prob_z[z] += probabilities[i, z]
        prob_z[z] /= len(probabilities)

    # p(y|z)
    prob_y_given_z=np.zeros((num_classes_target, num_classes_source))
    for y in range(num_classes_target):
        for z in range(num_classes_source):
            prob_y_given_z[y,z] = prob_y_z[y,z]/prob_z[z]

    # LEEP
    leep_score = 0.0
    non_null_sum=0
    for i in range(len(probabilities)):
        sum=0
        for z in range(num_classes_source):
            sum += prob_y_given_z[labels_target[i], z]*probabilities[i, z]
        if sum!=0:
            leep_score += np.log(sum)
            non_null_sum+=1
    leep_score /= non_null_sum
    return leep_score

input_shape = (64, 64, 3)
num_classes_source = 1000
num_classes_target = 30  

base_model = VGG16(include_top=False, weights='imagenet', input_shape=input_shape)
x = tf.keras.layers.Flatten()(base_model.output)
adapt_model = models.Sequential([
    layers.Flatten(name="flatten"),
    layers.Dense(4096, activation="relu", name="fc1"),
    layers.Dense(4096, activation="relu", name="fc2"),
    layers.Dense(num_classes_source, activation=None, name="predictions")
])
logits_layer = adapt_model(x)
pretrained_model = tf.keras.models.Model(inputs=base_model.input, outputs=logits_layer)

labels=[]
for lbl in labels_target:
    labels.append(lbl.numpy()[0])
leep = LEEP(pretrained_model, images_target, labels, num_classes_source, num_classes_target)
print("LEEP Score:", leep)

2024-07-04 13:59:39.986414: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
2024-07-04 13:59:50.327855: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
2024-07-04 14:00:00.387428: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
2024-07-04 14:00:10.714014: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
2024-07-04 14:00:20.711401: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
2024-07-04 14:00:20.720996: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:450] ShuffleDatasetV3:736: Filling up shuffle buffer (this may take a while): 1 of 20000
2024-07-04 14:00:30.721195: I tensorflow/core/kernels/data/shuffl

171/171 ━━━━━━━━━━━━━━━━━━━━ 7s 37ms/step


2024-07-04 14:05:02.713130: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
/usr/lib/python3.11/contextlib.py:158: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self.gen.throw(typ, value, traceback)


LEEP Score: -3.2632548046612047
